# Проект: книжные рекомендации

**Задача рекомендательной системы** — персонализировать рекомендации книг на основе поведения пользователей: их предпочтений по жанрам, дочитанным или дослушанным произведениям, факту прочтения, времени потребления контента, платформе использования и геолокации.

### Задача проекта:

-  построить ETL-пайплайн, который будет обрабатывать данные в пакетном режиме для работы рекомендательной системы.

Детали:

- ETL-пайплайн должен собирать **пять витрин** данных разной степени детализации для рекомендательной системы

### Описание витрин данных

#### user_content_mart

Решаемая задача:

- Обучение моделей рекомендаций разных типов (ranking, implicit feedback) на основе того, что пользователь уже читал или слушал.
- Расчёт признаков, основанных на взаимодействии пользователя с разными произведениями.


#### user_genre_mart

Решаемая задача:

- Построение профиля интересов пользователя.
- Рекомендации при «холодном старте» пользователя (англ. cold start scenario), то есть при отсутствии значительной истории данных.
- Генерация рекомендаций по жанрам.


#### user_mart

Решаемая задача:

- Хранение пользовательских фич, то есть определение профиля пользователя.
- Сегментация пользователей.


#### content_mart

Решаемая задача:

- Хранение контентных фич, то есть данных о популярности и качестве контента.
- Оценка популярности конкретных произведений.
- Создание сбалансированных и персонализированных рекомендаций, а не просто сводки популярного контента.


#### daily_user_mart

Решаемая задача:

- Определение динамики активности пользователей.
- Хранение признаков, основанных на временных паттернах.

### Исходные данные:

две таблицы в формате Parquet:

- <https://code.s3.yandex.net/data-scientist/dsplus/audition.parquet>,
- <https://code.s3.yandex.net/data-scientist/dsplus/content.parquet>

Ссылки для pyspark:

- s3a://s3-ds-source/audition.parquet
- s3a://s3-ds-source/content.parquet
- endpoint: <https://storage.yandexcloud.net>

In [ ]:
# импорты
import os
from pyspark.sql import SparkSession
from dotenv import load_dotenv

load_dotenv()

### Создание Spark-сессии и чтение данных

In [ ]:
def create_spark_session():
  """Функция создания Spark-сессии"""
  spark = (
      SparkSession.builder
        .appName('book_recommendations')
        .master("local[*]")
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4",
        )
        # endpoint
        .config("spark.hadoop.fs.s3a.endpoint", "https://storage.yandexcloud.net")
        # Ключ доступа
        .config("spark.hadoop.fs.s3a.access.key", os.environ["S3_ACCESS_KEY"])

        # Секретный ключ
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["S3_SECRET_KEY"])

        # Реализация файловой системы для работы с S3 через протокол S3A
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

        # Используем path-style access:
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.executor.memory", "2g")
        .config("spark.executor.cores", "2")
        .config("spark.executor.instances", "2")
        .config("spark.driver.cores", "4")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "32")
        .config("spark.network.timeout", "300")
        .getOrCreate()
  )
  return spark



In [ ]:
spark = create_spark_session()

### Первичное знакомство с данными

In [ ]:
def show_data_params(df):
    print('-'*30)
    
    print('Кол-во строк: ', df.count())
    print('Кол-во колонок: ', len(df.columns))
    df.printSchema()
    df.show(5, truncate=False)

In [ ]:
# загрузка
df_audition = spark.read.parquet('s3a://s3-ds-source/audition.parquet')
df_content = spark.read.parquet('s3a://s3-ds-source/content.parquet')

In [ ]:
# информация о датафреймах
show_data_params(df=df_audition)
show_data_params(df=df_content)

#### Анализ:

**Датафрейм audition**:
- Кол-во строк:  1002896, Кол-во колонок:  12

**Датафрейм content**:
- Кол-во строк:  31668, Кол-во колонок:  6


Общяя информация:
- все поля nullable
- явных ошибок в типах данных нет, данные соответствуют описанию, датасеты загружен корректно

